In [ ]:
!pip install -q torch transformers transformer_lens

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import torch
from transformer_lens.model_bridge import TransformerBridge

# Load GPT-2 Small
device = "cuda" if torch.cuda.is_available() else "cpu"

model = TransformerBridge.boot_transformers(
    "gpt2-medium",
    device=device
)

model.eval()

# Our experimental sentence
sentence = "John gave a bag to John."

# Tokenize
tokens = model.to_tokens(sentence)

print("Input sentence:", sentence)
print("Token IDs:", tokens)

# Inspect the individual GPT-2 tokens
print("\nToken representations:")

for i, token_id in enumerate(tokens[0]):
    print(i, int(token_id))

config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.52GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Input sentence: John gave a bag to John.
Token IDs: tensor([[50256,  7554,  2921,   257,  6131,   284,  1757,    13]])

Token representations:
0 50256
1 7554
2 2921
3 257
4 6131
5 284
6 1757
7 13


**Extracting the internal activations**

In [ ]:
# Run GPT-2 and cache its internal activations
with torch.no_grad():
    logits, cache = model.run_with_cache(sentence)

print("Available activation components:")

for name in cache.keys():
    if "attn" in name or "resid" in name or "mlp" in name:
        print(name)

Available activation components:
blocks.0.hook_resid_pre
blocks.0.attn.hook_in
blocks.0.attn.q.hook_in
blocks.0.attn.q.hook_out
blocks.0.attn.hook_q
blocks.0.attn.k.hook_in
blocks.0.attn.k.hook_out
blocks.0.attn.hook_k
blocks.0.attn.v.hook_in
blocks.0.attn.v.hook_out
blocks.0.attn.hook_v
blocks.0.attn.hook_attn_scores
blocks.0.attn.hook_pattern
blocks.0.attn.o.hook_in
blocks.0.attn.hook_z
blocks.0.attn.o.hook_out
blocks.0.attn.hook_hidden_states
blocks.0.attn.hook_out
blocks.0.hook_attn_out
blocks.0.hook_resid_mid
blocks.0.mlp.hook_in
blocks.0.mlp.in.hook_in
blocks.0.mlp.in.hook_out
blocks.0.mlp.hook_pre
blocks.0.mlp.out.hook_in
blocks.0.mlp.hook_post
blocks.0.mlp.out.hook_out
blocks.0.mlp.hook_out
blocks.0.hook_mlp_out
blocks.0.hook_resid_post
blocks.1.hook_resid_pre
blocks.1.attn.hook_in
blocks.1.attn.q.hook_in
blocks.1.attn.q.hook_out
blocks.1.attn.hook_q
blocks.1.attn.k.hook_in
blocks.1.attn.k.hook_out
blocks.1.attn.hook_k
blocks.1.attn.v.hook_in
blocks.1.attn.v.hook_out
blocks.1.a

In [ ]:
sentence = "John gave a bag to John."

tokens = model.to_tokens(sentence)

for i, token_id in enumerate(tokens[0]):
    token_id = int(token_id)

    print({
        "position": i,
        "token_id": token_id,
        "decoded": model.tokenizer.decode([token_id])
    })

{'position': 0, 'token_id': 50256, 'decoded': '<|endoftext|>'}
{'position': 1, 'token_id': 7554, 'decoded': 'John'}
{'position': 2, 'token_id': 2921, 'decoded': ' gave'}
{'position': 3, 'token_id': 257, 'decoded': ' a'}
{'position': 4, 'token_id': 6131, 'decoded': ' bag'}
{'position': 5, 'token_id': 284, 'decoded': ' to'}
{'position': 6, 'token_id': 1757, 'decoded': ' John'}
{'position': 7, 'token_id': 13, 'decoded': '.'}


**Let's grab the internal representation**

In [ ]:
layers_to_inspect = [0, 3, 6, 9, 12, 15, 18, 21, 23]

with torch.no_grad():
    logits, cache = model.run_with_cache(sentence)

for layer in layers_to_inspect:

    resid = cache[f"blocks.{layer}.hook_resid_post"]

    john1 = resid[0, 1]
    john2 = resid[0, 6]

    similarity = torch.nn.functional.cosine_similarity(
        john1.unsqueeze(0),
        john2.unsqueeze(0)
    )

    print(
        f"Layer {layer:02d}: "
        f"{similarity.item():.6f}"
    )

Layer 00: 0.975937
Layer 03: 0.959708
Layer 06: 0.940065
Layer 09: 0.908481
Layer 12: 0.886895
Layer 15: 0.892313
Layer 18: 0.914004
Layer 21: 0.947311
Layer 23: 0.954089


**L1H04**

An attention score of `0.4175` means that, **for this particular input**, this head assigns approximately `41.75%` of its attention probability to the first John when processing the second John.



In [ ]:
    import torch

with torch.no_grad():
    logits, cache = model.run_with_cache(sentence)

# Attention from second John (position 6)
# towards first John (position 1)

results = []

for layer in range(24):

    pattern = cache[
        f"blocks.{layer}.attn.hook_pattern"
    ]

    for head in range(16):

        score = pattern[0, head, 6, 1].item()

        results.append({
            "layer": layer,
            "head": head,
            "attention": score
        })

# Find the strongest connections
results.sort(
    key=lambda x: x["attention"],
    reverse=True
)

print("Top 15 attention connections:\n")

for item in results[:15]:
    print(item)

Top 15 attention connections:

{'layer': 1, 'head': 4, 'attention': 0.4175136387348175}
{'layer': 6, 'head': 4, 'attention': 0.3780008852481842}
{'layer': 7, 'head': 11, 'attention': 0.3675224483013153}
{'layer': 23, 'head': 5, 'attention': 0.29085344076156616}
{'layer': 0, 'head': 5, 'attention': 0.28878137469291687}
{'layer': 0, 'head': 10, 'attention': 0.2656073570251465}
{'layer': 4, 'head': 6, 'attention': 0.22300811111927032}
{'layer': 22, 'head': 2, 'attention': 0.18537795543670654}
{'layer': 2, 'head': 11, 'attention': 0.17744171619415283}
{'layer': 21, 'head': 7, 'attention': 0.17732416093349457}
{'layer': 1, 'head': 6, 'attention': 0.17044773697853088}
{'layer': 0, 'head': 0, 'attention': 0.15983858704566956}
{'layer': 15, 'head': 5, 'attention': 0.15863707661628723}
{'layer': 1, 'head': 15, 'attention': 0.15451978147029877}
{'layer': 14, 'head': 15, 'attention': 0.15175694227218628}


### **where does each head look?**
Before calling `L1H4` a meaningful circuit component, let's inspect its entire attention distribution.

For example, when the model processes the second John, does L1H4 attend primarily to:

The first `John`?

The verb `gave`?

The object `bag`?

The preposition `to`?

This will tell us more about what information the head might be collecting.

In [ ]:
candidate_heads = [
    (1, 4),
    (6, 4),
    (7, 11),
    (23, 5),
]

with torch.no_grad():
    logits, cache = model.run_with_cache(sentence)

tokens_list = model.to_str_tokens(sentence)

print("Token positions:")
for i, token in enumerate(tokens_list):
    print(i, repr(token))

for layer, head in candidate_heads:

    pattern = cache[
        f"blocks.{layer}.attn.hook_pattern"
    ]

    print(f"\nLayer {layer}, Head {head}")
    print("Attention from second John:")

    for position, token in enumerate(tokens_list):
        score = pattern[0, head, 6, position].item()

        print(
            f"Position {position}: "
            f"{token!r} = {score:.5f}"
        )

Token positions:
0 '<|endoftext|>'
1 'John'
2 ' gave'
3 ' a'
4 ' bag'
5 ' to'
6 ' John'
7 '.'

Layer 1, Head 4
Attention from second John:
Position 0: '<|endoftext|>' = 0.12575
Position 1: 'John' = 0.41751
Position 2: ' gave' = 0.00152
Position 3: ' a' = 0.01597
Position 4: ' bag' = 0.00038
Position 5: ' to' = 0.01306
Position 6: ' John' = 0.42580
Position 7: '.' = 0.00000

Layer 6, Head 4
Attention from second John:
Position 0: '<|endoftext|>' = 0.58046
Position 1: 'John' = 0.37800
Position 2: ' gave' = 0.00020
Position 3: ' a' = 0.00120
Position 4: ' bag' = 0.00000
Position 5: ' to' = 0.00020
Position 6: ' John' = 0.03994
Position 7: '.' = 0.00000

Layer 7, Head 11
Attention from second John:
Position 0: '<|endoftext|>' = 0.37568
Position 1: 'John' = 0.36752
Position 2: ' gave' = 0.00001
Position 3: ' a' = 0.00004
Position 4: ' bag' = 0.00000
Position 5: ' to' = 0.00003
Position 6: ' John' = 0.25671
Position 7: '.' = 0.00000

Layer 23, Head 5
Attention from second John:
Position 0: '

**This specific experiment reveals that L1H4 is sensitive to the repeated-name condition.**

Hence, L1H4 is the important component.

In [ ]:
import torch

sentences = [
    "John gave a bag to John.",
    "John gave a bag to Michael.",
    "Michael gave a bag to John."
]

candidate_heads = [
    (1, 4),
    (6, 4),
    (7, 11),
    (23, 5)
]

for text in sentences:

    tokens = model.to_str_tokens(text)

    print("\nSentence:", text)
    print("Tokens:", list(enumerate(tokens)))

    # Identify subject and recipient token positions.
    # This works when each name is represented by one token.
    subject = text.split()[0]
    recipient = text.rstrip(".").split()[-1]

    subject_positions = [
        i for i, token in enumerate(tokens)
        if i != 0 and token.strip() == subject
    ]

    recipient_positions = [
        i for i, token in enumerate(tokens)
        if i != 0 and token.strip() == recipient
    ]

    if not subject_positions or not recipient_positions:
        print("Name spans need manual inspection.")
        continue

    subject_pos = subject_positions[0]
    recipient_pos = recipient_positions[-1]

    with torch.no_grad():
        _, cache = model.run_with_cache(text)

    for layer, head in candidate_heads:

        pattern = cache[
            f"blocks.{layer}.attn.hook_pattern"
        ]

        score = pattern[
            0, head, recipient_pos, subject_pos
        ].item()

        print(
            f"L{layer}H{head}: "
            f"{score:.5f}"
        )


Sentence: John gave a bag to John.
Tokens: [(0, '<|endoftext|>'), (1, 'John'), (2, ' gave'), (3, ' a'), (4, ' bag'), (5, ' to'), (6, ' John'), (7, '.')]
L1H4: 0.41751
L6H4: 0.37800
L7H11: 0.36752
L23H5: 0.29085

Sentence: John gave a bag to Michael.
Tokens: [(0, '<|endoftext|>'), (1, 'John'), (2, ' gave'), (3, ' a'), (4, ' bag'), (5, ' to'), (6, ' Michael'), (7, '.')]
L1H4: 0.01892
L6H4: 0.00090
L7H11: 0.00001
L23H5: 0.26618

Sentence: Michael gave a bag to John.
Tokens: [(0, '<|endoftext|>'), (1, 'Michael'), (2, ' gave'), (3, ' a'), (4, ' bag'), (5, ' to'), (6, ' John'), (7, '.')]
L1H4: 0.01076
L6H4: 0.00277
L7H11: 0.00000
L23H5: 0.26483


| Attention Head | John → John | John → Michael | Michael → John |
| :--- | :---: | :---: | :---: |
| **L1H4** | 0.41751 | 0.01892 | 0.01076 |
| **L6H4** | 0.37800 | 0.00090 | 0.00277 |
| **L7H11** | 0.36752 | 0.00001 | 0.00000 |
| **L23H5** | 0.29085 | 0.26618 | 0.26483 |

**Generalising the repeated-name behaviour**

There are three possible explanations:

`Lexical matching:` L1H4 detects that the two names are the same.

`Token matching`: L1H4 responds to a particular relationship between token embeddings.

`Contextual matching`: L1H4 recognises a broader pattern involving the subject and recipient.

In [ ]:
import torch

test_sentences = [
    "John gave a bag to John.",
    "Alice gave a bag to Alice.",
    "Michael gave a bag to Michael.",
    "John gave a bag to Michael.",
    "Michael gave a bag to John."
]

layer = 1
head = 4

for text in test_sentences:

    tokens = model.to_str_tokens(text)

    print("\nSentence:", text)
    print("Tokens:", tokens)

    # Verify that the names occupy positions 1 and 6
    if len(tokens) != 8:
        print("Unexpected tokenisation. Skipping.")
        continue

    with torch.no_grad():
        _, cache = model.run_with_cache(text)

    pattern = cache[
        f"blocks.{layer}.attn.hook_pattern"
    ]

    score = pattern[0, head, 6, 1].item()

    print(f"L1H4 subject attention: {score:.6f}")


Sentence: John gave a bag to John.
Tokens: ['<|endoftext|>', 'John', ' gave', ' a', ' bag', ' to', ' John', '.']
L1H4 subject attention: 0.417514

Sentence: Alice gave a bag to Alice.
Tokens: ['<|endoftext|>', 'Alice', ' gave', ' a', ' bag', ' to', ' Alice', '.']
L1H4 subject attention: 0.406151

Sentence: Michael gave a bag to Michael.
Tokens: ['<|endoftext|>', 'Michael', ' gave', ' a', ' bag', ' to', ' Michael', '.']
L1H4 subject attention: 0.343887

Sentence: John gave a bag to Michael.
Tokens: ['<|endoftext|>', 'John', ' gave', ' a', ' bag', ' to', ' Michael', '.']
L1H4 subject attention: 0.018915

Sentence: Michael gave a bag to John.
Tokens: ['<|endoftext|>', 'Michael', ' gave', ' a', ' bag', ' to', ' John', '.']
L1H4 subject attention: 0.010758
